Build a Clean Dataset

This notebook demonstrates building the gold-layer dataset from clean silver data. It produces a **one-row-per-piece** parquet file with cumulative travel times, inter-stage partial times, and production context — segmented by die matrix.

### What this notebook does

1. Load clean pieces from `silver.clean_pieces` (all cleaning already applied)
2. Verify data quality: no zeros, no outliers, monotonic order, valid OEE
3. Compute partial times between process stages
4. Mark production gaps and assign run IDs
5. Inspect the final dataset structure per die matrix
6. Export to parquet (locally, or S3 when on AWS)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# -----------------------------
# Load environment variables
# -----------------------------
env_path = Path("../infra/.env")
if not env_path.exists():
    raise FileNotFoundError(f"Could not find env file: {env_path.resolve()}")

env = {}
for line in env_path.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    k, v = line.split("=", 1)
    env[k.strip()] = v.strip()

required = ["POSTGRES_PORT", "POSTGRES_DB", "POSTGRES_USER", "POSTGRES_PASSWORD"]
missing = [k for k in required if k not in env]
if missing:
    raise ValueError(f"Missing keys in ../infra/.env: {missing}")

# -----------------------------
# Database connection
# -----------------------------
engine = create_engine(
    f"postgresql+psycopg2://{env['POSTGRES_USER']}:{env['POSTGRES_PASSWORD']}"
    f"@localhost:{env['POSTGRES_PORT']}/{env['POSTGRES_DB']}"
)

with engine.connect() as conn:
    print("Connected to PostgreSQL:", conn.execute(text("SELECT current_database()")).scalar())

# -----------------------------
# Output path
# -----------------------------
gold_dir = Path("../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

gold_path = gold_dir / "pieces.parquet"
print("Gold parquet path:", gold_path.resolve())

Connected to PostgreSQL: vaultech
Gold parquet path: C:\Users\warre\VaultTech\data\gold\pieces.parquet


## 1. Load clean data from silver

The silver table contains only valid pieces — all signal-level noise, tracking failures, outliers, and monotonic violations were removed by the `01_bronze_to_silver` notebook.

In [2]:
df = pd.read_sql("""
SELECT *
FROM silver.clean_pieces
ORDER BY timestamp
""", engine)

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

print("silver.clean_pieces shape:", df.shape)
print("columns:", df.columns.tolist())
display(df.head(10))

silver.clean_pieces shape: (140404, 11)
columns: ['timestamp', 'piece_id', 'die_matrix', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_bath_s', 'lifetime_general_s', 'processed_at', 'oee_cycle_time_s', 'lifetime_auxiliary_press_s']


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_bath_s,lifetime_general_s,processed_at,oee_cycle_time_s,lifetime_auxiliary_press_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,56.200001,56.200001,2026-04-14 02:34:28.839803+00:00,NaN,54.599998
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,56.400002,56.400002,2026-04-14 02:34:28.839803+00:00,NaN,54.799999
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,56.900002,56.900002,2026-04-14 02:34:28.839803+00:00,NaN,55.299999
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,57.099998,57.099998,2026-04-14 02:34:28.839803+00:00,NaN,55.500000
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,57.200001,57.200001,2026-04-14 02:34:28.839803+00:00,NaN,55.500000
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,55.200001,55.200001,2026-04-14 02:34:28.839803+00:00,13.4690,53.599998
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,56.400002,56.400002,2026-04-14 02:34:28.839803+00:00,13.4496,54.700001
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,56.299999,56.299999,2026-04-14 02:34:28.839803+00:00,13.2704,54.700001
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,54.900002,54.900002,2026-04-14 02:34:28.839803+00:00,NaN,53.299999
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,55.799999,55.799999,2026-04-14 02:34:28.839803+00:00,NaN,54.099998


## 2. Verify data quality

Quick sanity check that the silver data is indeed clean — no zeros, no extreme values, monotonic order holds.

In [3]:
lifetime_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
]

lifetime_cols = [c for c in lifetime_cols if c in df.columns]

# 1) Zero check
zero_summary = pd.DataFrame({
    "column": lifetime_cols,
    "zero_count": [(df[c] == 0).sum() for c in lifetime_cols],
    "zero_pct": [((df[c] == 0).mean() * 100) for c in lifetime_cols],
})

# 2) Monotonic order check
monotonic_mask = (
    df["lifetime_2nd_strike_s"].notna()
    & df["lifetime_3rd_strike_s"].notna()
    & df["lifetime_4th_strike_s"].notna()
    & df["lifetime_auxiliary_press_s"].notna()
    & df["lifetime_bath_s"].notna()
    & (df["lifetime_2nd_strike_s"] < df["lifetime_3rd_strike_s"])
    & (df["lifetime_3rd_strike_s"] < df["lifetime_4th_strike_s"])
    & (df["lifetime_4th_strike_s"] < df["lifetime_auxiliary_press_s"])
    & (df["lifetime_auxiliary_press_s"] < df["lifetime_bath_s"])
)

# 3) OEE validity check
if "oee_cycle_time_s" in df.columns:
    invalid_oee_mask = (
        df["oee_cycle_time_s"].notna()
        & (
            (df["oee_cycle_time_s"] < 11)
            | (df["oee_cycle_time_s"] > 16)
        )
    )
else:
    invalid_oee_mask = pd.Series(False, index=df.index)

quality_summary = pd.DataFrame({
    "check": [
        "missing_timestamp",
        "duplicate_piece_id",
        "non_monotonic_rows",
        "invalid_oee_non_null_rows",
    ],
    "count": [
        df["timestamp"].isna().sum(),
        df["piece_id"].duplicated().sum() if "piece_id" in df.columns else np.nan,
        (~monotonic_mask).sum(),
        invalid_oee_mask.sum(),
    ],
})

print("Zero-value check")
display(zero_summary)

print("Quality summary")
display(quality_summary)

print("Non-monotonic rows:", int((~monotonic_mask).sum()))
print("Invalid non-null OEE rows:", int(invalid_oee_mask.sum()))


Zero-value check


,column,zero_count,zero_pct
0,lifetime_2nd_strike_s,0,0.0
1,lifetime_3rd_strike_s,0,0.0
2,lifetime_4th_strike_s,0,0.0
3,lifetime_auxiliary_press_s,0,0.0
4,lifetime_bath_s,0,0.0
5,lifetime_general_s,0,0.0


Quality summary


,check,count
0,missing_timestamp,0
1,duplicate_piece_id,845
2,non_monotonic_rows,0
3,invalid_oee_non_null_rows,0


Non-monotonic rows: 0
Invalid non-null OEE rows: 0


## 3. Dataset overview per die matrix

Each die matrix has different tooling geometry and expected travel times. All analysis must be segmented by matrix.

In [4]:
matrix_overview = (
    df.groupby("die_matrix")
    .agg(
        n_pieces=("piece_id", "nunique"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        median_2nd_strike=("lifetime_2nd_strike_s", "median"),
        median_3rd_strike=("lifetime_3rd_strike_s", "median"),
        median_4th_strike=("lifetime_4th_strike_s", "median"),
        median_aux_press=("lifetime_auxiliary_press_s", "median"),
        median_bath=("lifetime_bath_s", "median"),
        median_oee=("oee_cycle_time_s", "median"),
    )
    .round(2)
    .reset_index()
    .sort_values("n_pieces", ascending=False)
)

display(matrix_overview)


,die_matrix,n_pieces,first_timestamp,last_timestamp,median_2nd_strike,median_3rd_strike,median_4th_strike,median_aux_press,median_bath,median_oee
2,5090,81043,2025-12-04 21:06:17.707000+00:00,2026-02-17 13:31:14.137000+00:00,17.7,24.6,38.5,56.5,58.1,13.84
3,5091,22226,2025-11-25 18:37:19.931000+00:00,2026-02-26 00:57:01.946000+00:00,17.9,24.6,38.2,55.2,56.8,13.58
1,5052,20818,2025-11-06 15:25:16.426000+00:00,2026-02-25 03:45:42.232000+00:00,18.3,25.3,39.3,56.7,58.3,13.73
0,4974,15472,2025-11-13 20:19:47.883000+00:00,2025-11-25 09:25:29.704000+00:00,17.3,23.9,37.1,54.2,56.0,13.20


## 4. Compute partial times between stages

Since lifetime columns are cumulative from furnace exit, the time spent **between two consecutive stages** is computed by subtraction. All values in **seconds**.

| Partial column | Calculation | What it measures |
|---|---|---|
| `partial_furnace_to_2nd_strike_s` | `lifetime_2nd_strike_s` | Robot pick, transfer, positioning at main press |
| `partial_2nd_to_3rd_strike_s` | `lifetime_3rd - lifetime_2nd` | Press retraction, repositioning between strikes |
| `partial_3rd_to_4th_strike_s` | `lifetime_4th - lifetime_3rd` | Transfer to drill station on main press |
| `partial_4th_strike_to_auxiliary_press_s` | `lifetime_aux - lifetime_4th` | Exit main press, transfer to auxiliary press, deburring and coining |
| `partial_auxiliary_press_to_bath_s` | `lifetime_bath - lifetime_aux` | Transport from auxiliary press to quench bath |

In [5]:
df = df.sort_values("timestamp").reset_index(drop=True).copy()

df["partial_furnace_to_2nd_strike_s"] = df["lifetime_2nd_strike_s"]

df["partial_2nd_to_3rd_strike_s"] = (
    df["lifetime_3rd_strike_s"] - df["lifetime_2nd_strike_s"]
)

df["partial_3rd_to_4th_strike_s"] = (
    df["lifetime_4th_strike_s"] - df["lifetime_3rd_strike_s"]
)

df["partial_4th_strike_to_auxiliary_press_s"] = (
    df["lifetime_auxiliary_press_s"] - df["lifetime_4th_strike_s"]
)

df["partial_auxiliary_press_to_bath_s"] = (
    df["lifetime_bath_s"] - df["lifetime_auxiliary_press_s"]
)

partial_cols = [
    "partial_furnace_to_2nd_strike_s",
    "partial_2nd_to_3rd_strike_s",
    "partial_3rd_to_4th_strike_s",
    "partial_4th_strike_to_auxiliary_press_s",
    "partial_auxiliary_press_to_bath_s",
]

display(
    df[
        [
            "timestamp",
            "piece_id",
            "die_matrix",
            "lifetime_2nd_strike_s",
            "lifetime_3rd_strike_s",
            "lifetime_4th_strike_s",
            "lifetime_auxiliary_press_s",
            "lifetime_bath_s",
        ] + partial_cols
    ].head(10)
)


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,18.400000,6.700001,13.300001,17.099998,1.599998
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,18.200001,6.599998,13.400002,17.299999,1.700001
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,53.599998,55.200001,16.700001,6.699999,13.400000,16.799999,1.600002
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,54.700001,56.400002,18.299999,6.600000,13.300001,16.500000,1.700001
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,54.700001,56.299999,17.900000,6.600000,13.200001,17.000000,1.599998
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,53.299999,54.900002,16.700001,6.599998,13.299999,16.700001,1.600002
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,54.099998,55.799999,16.700001,6.900000,13.199999,17.299999,1.700001


## 5. Partial times per die matrix

Each matrix has its own expected timing profile. These medians serve as the **reference behavior** for deviation detection.

In [7]:
partial_stats_by_matrix = (
    df.groupby("die_matrix")[partial_cols]
    .agg(["count", "median", "mean", "std", "min", "max"])
    .round(2)
)

display(partial_stats_by_matrix)

partial_medians_by_matrix = (
    df.groupby("die_matrix")[partial_cols]
    .median()
    .round(2)
    .reset_index()
)

display(partial_medians_by_matrix)


partial_furnace_to_2nd_strike_s                                  \
                                     count median   mean   std   min   max   
die_matrix                                                                   
4974                                 15531   17.3  17.76  1.58  11.2  23.9   
5052                                 20887   18.3  18.61  1.91  10.5  26.6   
5090                                 81677   17.7  18.49  2.40   9.4  30.4   
5091                                 22309   17.9  18.69  2.45   9.7  31.0   

           partial_2nd_to_3rd_strike_s                     ...  \
                                 count median  mean   std  ...   
die_matrix                                                 ...   
4974                             15531    6.5  6.57  0.27  ...   
5052                             20887    6.9  6.95  0.50  ...   
5090                             81677    6.8  6.96  0.74  ...   
5091                             22309    6.6  6.78  0.75  ...   

           partial_4th_strike_to_auxiliary_press_s                   \
                                              mean   std  min   max   
die_matrix                                                            
4974                                         17.06  0.88  8.3  25.7   
5052                                         17.34  1.15  8.3  31.1   
5090                                         17.62  1.62  7.8  34.8   
5091                                         17.04  1.04  8.1  34.8   

           partial_auxiliary_press_to_bath_s                                
                                       count median  mean   std  min   max  
die_matrix                                                                  
4974                                   15531    1.8  1.76  0.05  1.7   1.9  
5052                                   20887    1.6  1.64  0.05  1.5   1.7  
5090                                   81677    1.6  1.62  0.09  1.4  18.2  
5091                                   22309    1.6  1.63  0.11  1.5  17.0  

[4 rows x 30 columns]

,die_matrix,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s
0,4974,17.3,6.5,13.1,17.0,1.8
1,5052,18.3,6.9,13.7,17.3,1.6
2,5090,17.7,6.8,13.8,17.7,1.6
3,5091,17.9,6.6,13.5,17.0,1.6


## 6. Mark production gaps

Flag pieces that follow a production gap (> 5 minutes). Assign a `production_run_id` to group consecutive pieces within the same uninterrupted run.

In [8]:
df = df.sort_values("timestamp").reset_index(drop=True).copy()

df["seconds_since_prev_piece"] = df["timestamp"].diff().dt.total_seconds()

df["is_production_gap"] = df["seconds_since_prev_piece"] > 300
df["is_production_gap"] = df["is_production_gap"].fillna(False)

df["production_run_id"] = df["is_production_gap"].cumsum() + 1

print("Number of production gaps:", int(df["is_production_gap"].sum()))
print("Number of production runs:", int(df["production_run_id"].nunique()))

display(
    df[
        [
            "timestamp",
            "piece_id",
            "die_matrix",
            "seconds_since_prev_piece",
            "is_production_gap",
            "production_run_id",
        ]
    ].head(20)
)

Number of production gaps: 775
Number of production runs: 776


,timestamp,piece_id,die_matrix,seconds_since_prev_piece,is_production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,NaN,False,1
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,12.708,False,1
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,14.609,False,1
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,12.719,False,1
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,14.000,False,1
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,13.309,False,1
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,12.611,False,1
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,13.713,False,1
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,67.332,False,1
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,67.043,False,1


## 7. Final dataset structure

The gold dataset has one row per piece with all identification, cumulative times, partial times, OEE, and production context.

In [9]:
final_cols = [
    "timestamp",
    "piece_id",
    "die_matrix",
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
    "partial_furnace_to_2nd_strike_s",
    "partial_2nd_to_3rd_strike_s",
    "partial_3rd_to_4th_strike_s",
    "partial_4th_strike_to_auxiliary_press_s",
    "partial_auxiliary_press_to_bath_s",
    "oee_cycle_time_s",
    "seconds_since_prev_piece",
    "is_production_gap",
    "production_run_id",
]

final_cols = [c for c in final_cols if c in df.columns]
gold_df = df[final_cols].copy()

print("Final gold dataset shape:", gold_df.shape)
print("Final columns:")
print(gold_df.columns.tolist())

display(gold_df.head(10))


Final gold dataset shape: (140404, 18)
Final columns:
['timestamp', 'piece_id', 'die_matrix', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_bath_s', 'lifetime_general_s', 'partial_furnace_to_2nd_strike_s', 'partial_2nd_to_3rd_strike_s', 'partial_3rd_to_4th_strike_s', 'partial_4th_strike_to_auxiliary_press_s', 'partial_auxiliary_press_to_bath_s', 'oee_cycle_time_s', 'seconds_since_prev_piece', 'is_production_gap', 'production_run_id']


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s,oee_cycle_time_s,seconds_since_prev_piece,is_production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002,NaN,NaN,False,1
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002,NaN,12.708,False,1
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002,NaN,14.609,False,1
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,18.400000,6.700001,13.300001,17.099998,1.599998,NaN,12.719,False,1
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,18.200001,6.599998,13.400002,17.299999,1.700001,NaN,14.000,False,1
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,53.599998,55.200001,55.200001,16.700001,6.699999,13.400000,16.799999,1.600002,13.4690,13.309,False,1
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,54.700001,56.400002,56.400002,18.299999,6.600000,13.300001,16.500000,1.700001,13.4496,12.611,False,1
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,54.700001,56.299999,56.299999,17.900000,6.600000,13.200001,17.000000,1.599998,13.2704,13.713,False,1
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,53.299999,54.900002,54.900002,16.700001,6.599998,13.299999,16.700001,1.600002,NaN,67.332,False,1
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,54.099998,55.799999,55.799999,16.700001,6.900000,13.199999,17.299999,1.700001,NaN,67.043,False,1


## 8. Export to parquet

Save the gold dataset. When running on AWS, change `GOLD_DIR` to an S3 path.

In [10]:
from pathlib import Path

# -----------------------------
# Configure output path
# -----------------------------
GOLD_DIR = Path("../data/gold")   # Local
# GOLD_DIR = "s3://your-bucket/gold"  # AWS version (just change this)

# -----------------------------
# Save parquet
# -----------------------------
if isinstance(GOLD_DIR, Path):
    GOLD_DIR.mkdir(parents=True, exist_ok=True)
    output_path = GOLD_DIR / "pieces.parquet"
else:
    output_path = f"{GOLD_DIR}/pieces.parquet"

gold_df.to_parquet(output_path, index=False)

print("Parquet saved to:", output_path)
print("Rows:", len(gold_df))
print("Columns:", len(gold_df.columns))

# -----------------------------
# Reload to verify (quality gate)
# -----------------------------
gold_check = pd.read_parquet(output_path)

print("Reloaded shape:", gold_check.shape)
print("Columns match:", list(gold_check.columns) == list(gold_df.columns))

display(gold_check.head(5))


Parquet saved to: ..\data\gold\pieces.parquet
Rows: 140404
Columns: 18
Reloaded shape: (140404, 18)
Columns match: True


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s,oee_cycle_time_s,seconds_since_prev_piece,is_production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002,NaN,NaN,False,1
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002,NaN,12.708,False,1
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002,NaN,14.609,False,1
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,18.400000,6.700001,13.300001,17.099998,1.599998,NaN,12.719,False,1
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,18.200001,6.599998,13.400002,17.299999,1.700001,NaN,14.000,False,1
